In [ ]:
!python -m pip install faiss-cpu

In [ ]:
chunks = [
    "Artificial intelligence is transforming healthcare through predictive analytics.",
    "Machine learning algorithms improve recommendation systems in e-commerce.",
    "Natural language processing enables chatbots to understand human language.",
    "Cloud computing provides scalable infrastructure for modern applications.",
    "Cybersecurity protects systems from unauthorized access and cyber threats.",
    "Data science combines statistics, programming, and domain expertise.",
    "Deep learning models require large datasets for effective training.",
    "Vector databases are optimized for similarity search using embeddings.",
    "FAISS is a popular library for approximate nearest neighbor search.",
    "HNSW constructs a graph structure for efficient vector retrieval.",
    "Python is widely used for machine learning and data analysis.",
    "SQL databases store structured data using relational tables.",
    "NoSQL databases provide flexibility for semi-structured information.",
    "Docker simplifies application deployment using containers.",
    "Kubernetes automates container orchestration in cloud environments.",
    "Large language models generate human-like text responses.",
    "Semantic search retrieves documents based on meaning rather than keywords.",
    "Embeddings convert text into dense numerical vectors.",
    "Retrieval augmented generation combines search with language models.",
    "Transformers revolutionized natural language processing tasks.",
    "Computer vision enables machines to interpret digital images.",
    "Speech recognition converts spoken language into text.",
    "Reinforcement learning trains agents through rewards and penalties.",
    "Distributed systems improve scalability and fault tolerance.",
    "Graph databases represent relationships between entities efficiently.",
    "Apache Spark processes large-scale data in distributed environments.",
    "TensorFlow is a framework for deep learning applications.",
    "PyTorch is widely used for research in artificial intelligence.",
    "Neural networks consist of interconnected layers of artificial neurons.",
    "Principal component analysis reduces data dimensionality.",
    "Clustering groups similar data points without labels.",
    "Classification predicts discrete categories for input samples.",
    "Regression estimates continuous numerical values.",
    "Decision trees provide interpretable machine learning models.",
    "Random forests combine multiple decision trees for better accuracy.",
    "Gradient boosting builds strong predictive models iteratively.",
    "Support vector machines classify data using optimal hyperplanes.",
    "Cross-validation estimates model performance on unseen data.",
    "Feature engineering improves machine learning performance.",
    "Hyperparameter tuning optimizes model configurations.",
    "Cosine similarity measures the angle between vectors.",
    "Euclidean distance measures straight-line distance in space.",
    "Approximate nearest neighbor search trades accuracy for speed.",
    "Indexing accelerates search operations on large datasets.",
    "Caching reduces response times by storing frequent results.",
    "Load balancing distributes traffic across multiple servers.",
    "Microservices divide applications into independent services.",
    "REST APIs enable communication between software systems.",
    "GraphQL provides flexible querying for APIs.",
    "Data warehouses support analytical workloads."
]
len(chunks)

# Download Embedding Model and Initialize it  

In [ ]:
!python -m pip install langchain-ollama

In [ ]:
!ollama list 

In [ ]:
from langchain_ollama import OllamaEmbeddings
emb_model = OllamaEmbeddings(model="embeddinggemma-2:latest")

# Create Embeddings for all the chunks 

In [ ]:
import numpy as np 
chunk_embeddings = emb_model.embed_documents(chunks)
chunk_embeddings = np.array(chunk_embeddings, dtype="float32")
print(type(chunk_embeddings), chunk_embeddings.shape)

# Create FAISS Index 

In [ ]:
import faiss
import pickle

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

# Add data to index 
index.add(chunk_embeddings)

# Save Index to file 
index_path = "./Flat_Index/flat_index.index"
faiss.write_index(index, index_path)
print("Index Saved in :", index_path)

chunk_path = "./Flat_Index/flat_chunks.pkl"
with open(chunk_path, "wb") as f:
    pickle.dump(chunks, f)
print("Chunks Saved in :", chunk_path)

In [ ]:
# Load index and Chunks 
index_path = "./Flat_Index/flat_index.index"
index = faiss.read_index(index_path)

chunk_path = "./Flat_Index/flat_chunks.pkl"
with open(chunk_path, "rb") as fl:
    documents = pickle.load(fl)


In [ ]:
query = "Tell me about regression and SVM models"
query_embeddings = emb_model.embed_documents(query)
query_embeddings = np.array(query_embeddings, dtype="float32")
query_embeddings.shape

In [ ]:
distances, doc_index = index.search(query_embeddings, k=2)
doc_index = doc_index[0]
distances = distances[0]
print(f"{distances=}, {doc_index=}")

retrived_chunks  = []
for ind in doc_index:
    retrived_chunks.append(documents[ind])
print(retrived_chunks)